In [145]:
"""
Your dataset contains ~40,000 reservations for a 200-room resort hotel in Portugal, 
including the booking date, 
check-in & check-out dates, and a cancellation flag.
"""

'\nYour dataset contains ~40,000 reservations for a 200-room resort hotel in Portugal, \nincluding the booking date, \ncheck-in & check-out dates, and a cancellation flag.\n'

In [146]:
import pandas as pd

In [147]:
hotel_bookings_df = pd.read_csv("hotel_bookings.csv", parse_dates=['booking_date', 'cancel_date' ,'checkin_date', 'checkout_date'])
hotel_bookings_df.head()

,booking_id,booking_date,cancel_date,checkin_date,checkout_date,is_canceled
0,1,2014-03-18,NaT,2016-02-25,2016-03-24,0
1,2,2014-04-18,NaT,2015-10-02,2015-10-11,0
2,3,2014-04-30,NaT,2015-08-03,2015-08-10,0
3,4,2014-04-30,NaT,2015-08-03,2015-08-10,0
4,5,2014-04-30,NaT,2015-08-03,2015-08-10,0


In [148]:
# filter for no canceled bookings
hotel_bookings_no_cancel = hotel_bookings_df[hotel_bookings_df['is_canceled'] == 0].copy()
hotel_bookings_no_cancel.drop(columns=['cancel_date'], inplace=True)
hotel_bookings_no_cancel.head()

,booking_id,booking_date,checkin_date,checkout_date,is_canceled
0,1,2014-03-18,2016-02-25,2016-03-24,0
1,2,2014-04-18,2015-10-02,2015-10-11,0
2,3,2014-04-30,2015-08-03,2015-08-10,0
3,4,2014-04-30,2015-08-03,2015-08-10,0
4,5,2014-04-30,2015-08-03,2015-08-10,0


In [149]:
hotel_bookings_no_cancel['night'] = [pd.date_range(checkin_date, checkout_date, inclusive='left') for checkin_date, checkout_date in 
              zip(hotel_bookings_no_cancel.checkin_date, 
            hotel_bookings_no_cancel.checkout_date)]
hotel_bookings_no_cancel.head()

,booking_id,booking_date,checkin_date,checkout_date,is_canceled,night
0,1,2014-03-18,2016-02-25,2016-03-24,0,"DatetimeIndex(['2016-02-25', '2016-02-26', '20..."
1,2,2014-04-18,2015-10-02,2015-10-11,0,"DatetimeIndex(['2015-10-02', '2015-10-03', '20..."
2,3,2014-04-30,2015-08-03,2015-08-10,0,"DatetimeIndex(['2015-08-03', '2015-08-04', '20..."
3,4,2014-04-30,2015-08-03,2015-08-10,0,"DatetimeIndex(['2015-08-03', '2015-08-04', '20..."
4,5,2014-04-30,2015-08-03,2015-08-10,0,"DatetimeIndex(['2015-08-03', '2015-08-04', '20..."


In [150]:
df_date = hotel_bookings_no_cancel.explode('night')
df_date.head()

,booking_id,booking_date,checkin_date,checkout_date,is_canceled,night
0,1,2014-03-18,2016-02-25,2016-03-24,0,2016-02-25
0,1,2014-03-18,2016-02-25,2016-03-24,0,2016-02-26
0,1,2014-03-18,2016-02-25,2016-03-24,0,2016-02-27
0,1,2014-03-18,2016-02-25,2016-03-24,0,2016-02-28
0,1,2014-03-18,2016-02-25,2016-03-24,0,2016-02-29


In [157]:
df_month = df_date.night.dt.to_period('M').value_counts().sort_index().to_frame('Booked')
df_month.head()

,Booked
night,
2015-07,4944
2015-08,5557
2015-09,5428
2015-10,4829
2015-11,3295


In [158]:
df_month['Capacity'] = df_month.index.days_in_month * 200
df_month.head()

,Booked,Capacity
night,,
2015-07,4944,6200
2015-08,5557,6200
2015-09,5428,6000
2015-10,4829,6200
2015-11,3295,6000


In [153]:
df_month['Occupancy'] = 100 * df_month['Booked'] / df_month['Capacity']
df_month.head()

,Booked,Capacity,Occupancy
night,,,
2015-07,4944,6200,79.741935
2015-08,5557,6200,89.629032
2015-09,5428,6000,90.466667
2015-10,4829,6200,77.887097
2015-11,3295,6000,54.916667


In [154]:
# Flag for july 2016
df_month[df_month.index == '2016-07']

,Booked,Capacity,Occupancy
night,,,
2016-07,5491,6200,88.564516
